In [6]:
import json
from pathlib import Path
import torch
import causal_conv1d
import fla
import accelerate
import transformers
import sys
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_PATH = "/workspace/models/Qwen3.8-27B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    local_files_only=True,
)

model.eval()

In [31]:
PROJECT_ROOT = Path("/workspace/storyfx")

excerpt = "excerpt_001"
task = "ambient_sfx_specification"
strategy = "storyfx"

print(f"Input Excerpt Text: {excerpt}")
print(f"Composition Planning Task: {task}")
print(f"Selected Prompting Strategy: {strategy}")

Input Excerpt Text: excerpt_001
Composition Planning Task: ambient_sfx_specification
Selected Prompting Strategy: storyfx


In [32]:
prompt_path = (
    PROJECT_ROOT
    / "configs"
    / "prompts"
    / strategy
    / f"{task}.txt"
)

excerpt_path = (
    PROJECT_ROOT
    / "data"
    / excerpt
    / "raw"
    / f"{excerpt}.txt"
)

output_path = (
    PROJECT_ROOT
    / "outputs"
    / "composition_plans"
    / f"pred_{excerpt}_{task}.json"
)

print(f"Prompt Path: {prompt_path}")
print(f"Excerpt Path: {excerpt_path}")
print(f"Output Path: {output_path}")

Prompt Path: /workspace/storyfx/configs/prompts/storyfx/ambient_sfx_specification.txt
Excerpt Path: /workspace/storyfx/data/excerpt_001/raw/excerpt_001.txt
Output Path: /workspace/storyfx/outputs/composition_plans/pred_excerpt_001_ambient_sfx_specification.json


In [33]:
def extract_dramatized_segments(segment_spec_path):
    segment_spec_path = Path(segment_spec_path)

    segments = json.loads(
        segment_spec_path.read_text(encoding="utf-8")
    )

    dramatized_segments = [
        {
            "segment_id": segment["segment_id"],
            "start": segment["start"],
            "end": segment["end"],
        }
        for segment in segments
        if segment.get("segment_type") == "dramatized"
    ]

    return dramatized_segments

In [34]:
prompt_template = prompt_path.read_text(
    encoding="utf-8"
).strip()

excerpt_text = excerpt_path.read_text(
    encoding="utf-8"
).strip()

if task == "ambient_sfx_specification":

    segment_spec_path = (
        PROJECT_ROOT
        / "outputs"
        / "composition_plans"
        / f"pred_{excerpt}_segment_specification.json"
    )

    dramatized_segments = extract_dramatized_segments(
        segment_spec_path
    )

    segments_text = json.dumps(
        dramatized_segments,
        indent=2,
        ensure_ascii=False,
    )

    prompt = (
        f"{prompt_template}\n\n"
        f"### INPUT NARRATIVE\n\n"
        f"{excerpt_text}\n\n"
        f"### INPUT SEGMENTS\n\n"
        f"{segments_text}"
    )

else:

    prompt = (
        f"{prompt_template}\n\n"
        f"### INPUT NARRATIVE\n\n"
        f"{excerpt_text}"
    )

messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

In [35]:
def strip_json_markup(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    elif text.startswith("```"):
        text = text[3:].strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text

In [36]:
ENABLE_THINKING = False
MAX_NEW_TOKENS = 3000


inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=ENABLE_THINKING,
).to(model.device)


if ENABLE_THINKING:
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": True,
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 20,
    }
else:
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
    }


with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        **generation_kwargs,
    )


generated_ids = outputs[0][
    inputs["input_ids"].shape[-1]:
]

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True,
).strip()


# Remove Qwen thinking trace if any before parsing JSON
if "</think>" in response:
    final_response = response.split("</think>", 1)[1].strip()
else:
    final_response = response

final_response=strip_json_markup(final_response)

print(final_response)


parsed_response = json.loads(final_response)


output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

output_path.write_text(
    json.dumps(
        parsed_response,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(f"Specification saved to: {output_path}")

[
  {
    "segment_id": "segment_001",
    "place": null,
    "weather": "windy rainstorm weather outside",
    "special": {
      "caption": null,
      "looping_behavior": null
    }
  },
  {
    "segment_id": "segment_002",
    "place": null,
    "weather": "windy rainstorm weather",
    "special": {
      "caption": null,
      "looping_behavior": null
    }
  },
  {
    "segment_id": "segment_003",
    "place": null,
    "weather": "muffled windy rain weather outside",
    "special": {
      "caption": null,
      "looping_behavior": null
    }
  },
  {
    "segment_id": "segment_004",
    "place": null,
    "weather": "muffled windy rain weather outside",
    "special": {
      "caption": "recurring supernatural whispers from walls",
      "looping_behavior": "intermittent"
    }
  },
  {
    "segment_id": "segment_005",
    "place": null,
    "weather": "muffled windy rain weather outside",
    "special": {
      "caption": null,
      "looping_behavior": null
    }
  }
]
Specif